# Full Run — Visual Encoding of Importance in Paintings

Applies the techniques locked in by the Renaissance pilot (see thesis §4.3, §4.8)
to the 200-painting balanced full-run set.

**Locked-in choices** (from `pilot_conclusion.csv`):

- Person detection: YOLOv7, person class, confidence 0.25
- Validation gate: IoU $\geq 0.20$ between hand-drawn important box and YOLO detection
- Segmentation: SAM3, prompted by YOLO boxes
- Saliency: DIS
- Isolation feature: `iso_nearest_centroid_norm` (higher = more isolated)
- Elevation feature: `elev_bbox_top` (higher = higher in the frame)
- Salience feature: `sal_sum` (higher = more salient)
- Primary score: `combo_saliency_heavy` = $0.50 \cdot S + 0.25 \cdot I + 0.25 \cdot E$ (each min-max normalised within painting)
- Secondary score: `sal_high_coverage_50`

This notebook is executed by `full_run_orchestrator.py` once YOLO, SAM3, and DIS
have produced their outputs. Re-running it manually is fine — paths are read from
environment variables with sensible fallbacks.

## Setup

In [ ]:
import os
import sys
import json
import gc
from pathlib import Path
from functools import reduce

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist

# Large Renaissance / Baroque paintings can exceed Pillow's default safety limit.
Image.MAX_IMAGE_PIXELS = None

# Local imports — gate.py lives next to this notebook.
sys.path.insert(0, str(Path.cwd()))
import gate as gate_module

### Paths

In [ ]:
# Paths — read from environment when the orchestrator runs us, otherwise fall back
# to the conventional layout so the notebook is runnable interactively too.
HOME = Path.home()
DEFAULT_RUN  = HOME / 'Thesis' / 'full_run'
DEFAULT_DATA = HOME / 'Thesis' / 'Images' / 'full run images balanced'  

def env_path(key, default):
    return Path(os.environ.get(key, str(default)))

IMAGES_DIR    = env_path('PIPELINE_IMAGES',      DEFAULT_RUN / 'outputs' / 'images_subset')
ANNOTATIONS   = env_path('PIPELINE_ANNOTATIONS', DEFAULT_DATA / 'annotations.json')
YOLO_LABELS   = env_path('PIPELINE_YOLO_LABELS', DEFAULT_RUN / 'outputs' / 'yolov7' / 'persons_only_025' / 'labels')
SAM3_ROOT     = env_path('PIPELINE_SAM_OUT',     DEFAULT_RUN / 'outputs' / 'sam3')
DIS_DIR       = env_path('PIPELINE_DIS_OUT',     DEFAULT_RUN / 'outputs' / 'dis')
OUTPUTS       = env_path('PIPELINE_OUTPUTS',     DEFAULT_RUN / 'outputs')

IOU_THRESHOLD = float(os.environ.get('PIPELINE_IOU_THRESHOLD', '0.20'))

# Output sub-folders.
FEATURE_DIR = OUTPUTS / 'features_per_painting'
SUMMARY_DIR = OUTPUTS / 'summary'
for d in (FEATURE_DIR, SUMMARY_DIR):
    d.mkdir(parents=True, exist_ok=True)

for label, p in [
    ('IMAGES_DIR',   IMAGES_DIR),
    ('ANNOTATIONS',  ANNOTATIONS),
    ('YOLO_LABELS',  YOLO_LABELS),
    ('SAM3_ROOT',    SAM3_ROOT),
    ('DIS_DIR',      DIS_DIR),
    ('OUTPUTS',      OUTPUTS),
]:
    print(f'{label:14s} {p}   exists: {p.exists()}')
print(f'\nIoU threshold: {IOU_THRESHOLD}')

# 1. Validation Gate

For every painting:
1. load YOLOv7 person-class detections
2. compute IoU between each hand-drawn important box and every YOLO detection
3. the painting passes if at least one important box achieves IoU $\geq$ 0.20

Paintings that fail are logged as detection failures and excluded from feature
extraction — there is no point measuring the saliency or isolation of a figure the
detector never found.

In [ ]:
gate_df, paintings_passed, paintings_failed = gate_module.compute_gate(
    via_json_path   = ANNOTATIONS,
    yolo_labels_dir = YOLO_LABELS,
    images_dir      = IMAGES_DIR,
    iou_threshold   = IOU_THRESHOLD,
)

print(f'IoU threshold: {IOU_THRESHOLD}')
print(f'Passed: {len(paintings_passed)} paintings')
print(f'Failed: {len(paintings_failed)} paintings')

gate_df.to_csv(OUTPUTS / 'iou_per_painting.csv', index=False)
gate_df[~gate_df['passed_gate']].to_csv(OUTPUTS / 'detection_failures.csv', index=False)

print('\nGate result (head):')
display(gate_df.head(15).round(3))

In [ ]:
# Histogram of per-painting best-IoU, split by gate outcome.
fig, ax = plt.subplots(figsize=(9, 4.5))
bins = np.linspace(0, 1, 21)
ax.hist(gate_df.loc[gate_df['passed_gate'], 'any_important_iou'], bins=bins,
        alpha=0.75, color='#3b8edb', label=f'passed (n={len(paintings_passed)})')
ax.hist(gate_df.loc[~gate_df['passed_gate'], 'any_important_iou'], bins=bins,
        alpha=0.75, color='#d04646', label=f'failed (n={len(paintings_failed)})')
ax.axvline(IOU_THRESHOLD, color='black', linestyle='--', linewidth=1.2,
           label=f'threshold = {IOU_THRESHOLD}')
ax.set_xlabel('Best IoU (hand-drawn important box vs. any YOLO detection)')
ax.set_ylabel('Number of paintings')
ax.set_title('Per-painting best-IoU distribution')
ax.legend()
plt.tight_layout()
plt.savefig(SUMMARY_DIR / 'iou_distribution.png', dpi=200, bbox_inches='tight')
plt.show()

# 2. Feature Extraction (locked-in techniques)

For each gate-passing painting we compute the three locked-in features and nothing
else. SAM3 masks are loaded from `SAM3_ROOT/<painting_id>/mask_*.png`, the DIS
saliency map from `DIS_DIR/<painting_id>.png`.

- `iso_nearest_centroid_norm` — nearest-neighbour centroid distance, normalised by image diagonal
- `elev_bbox_top` — elevation of the top of the mask's tight bounding box, $1 - y/H$
- `sal_sum` — total DIS saliency inside the mask
- `sal_high_coverage_50` — secondary technique, also reported for the locked-in evaluation

In [ ]:
# ---- low-level helpers ----------------------------------------------------
def load_image_rgb(path):
    img = cv2.imread(str(path))
    if img is None:
        raise ValueError(f'Could not load image: {path}')
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def load_binary_mask(path, target_shape):
    H, W = target_shape
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise ValueError(f'Could not load mask: {path}')
    if mask.shape != (H, W):
        mask = cv2.resize(mask, (W, H), interpolation=cv2.INTER_NEAREST)
    thresh = 0 if mask.max() <= 1 else 127
    binary = mask > thresh
    # SAM3 sometimes returns inverted masks - flip when foreground covers most of the image.
    if binary.mean() > 0.5:
        binary = ~binary
    return binary


def _centroid_bbox_area(mask):
    """Centroid + tight bbox + area via cv2 moments. Fast for large masks."""
    mask_u8 = mask.astype(np.uint8)
    M = cv2.moments(mask_u8, binaryImage=True)
    if M['m00'] == 0:
        return (np.nan, np.nan), [np.nan, np.nan, np.nan, np.nan], 0
    cx = M['m10'] / M['m00']
    cy = M['m01'] / M['m00']
    x, y, w, h = cv2.boundingRect(mask_u8)
    bbox = [int(x), int(y), int(x + w - 1), int(y + h - 1)]
    return (float(cx), float(cy)), bbox, int(M['m00'])


def load_saliency_map(path, target_shape):
    sal = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if sal is None:
        raise ValueError(f'Could not load saliency map: {path}')
    H, W = target_shape
    if sal.shape != (H, W):
        sal = cv2.resize(sal, (W, H), interpolation=cv2.INTER_LINEAR)
    sal = sal.astype(np.float32)
    mn, mx = float(sal.min()), float(sal.max())
    if mx == mn:
        return np.zeros_like(sal, dtype=np.float32)
    return (sal - mn) / (mx - mn)


def find_dis_for_painting(dis_dir, painting_id):
    """Locate the DIS saliency output for a painting."""
    dis_dir = Path(dis_dir)
    if not dis_dir.exists():
        return None
    for ext in ('.png', '.jpg', '.jpeg'):
        p = dis_dir / f'{painting_id}{ext}'
        if p.exists():
            return p
    for p in dis_dir.rglob(f'{painting_id}.*'):
        if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}:
            return p
    return None

In [ ]:
# ---- feature extraction for one painting ----------------------------------
def extract_figures(mask_dir, image_shape):
    """Read all SAM3 masks in mask_dir, return ordered list of figures."""
    H, W = image_shape
    figures = []
    for mask_path in sorted(Path(mask_dir).glob('*.png')):
        mask = load_binary_mask(mask_path, (H, W))
        if not np.any(mask):
            continue
        (cx, cy), bbox, area = _centroid_bbox_area(mask)
        figures.append({
            'mask_file':  mask_path.name,
            'mask':       mask,
            'bbox':       bbox,
            'centroid_x': cx,
            'centroid_y': cy,
            'mask_area':  area,
        })
    # Deterministic left-to-right ordering, ties broken by y.
    figures.sort(key=lambda f: (f['centroid_x'], f['centroid_y']))
    for i, f in enumerate(figures, start=1):
        f['figure_id'] = f'fig_{i:03d}'
    return figures


def compute_painting_features(painting_id, figures, saliency_map, image_shape):
    """Compute the three locked-in features + the secondary salience coverage."""
    H, W = image_shape
    diag = float(np.sqrt(H ** 2 + W ** 2))
    centers = np.array([[f['centroid_x'], f['centroid_y']] for f in figures], dtype=float)

    rows = []
    for i, fig in enumerate(figures):
        # --- isolation: nearest-neighbour centroid distance ---
        if len(centers) > 1 and not np.isnan(centers[i]).any():
            others = np.delete(centers, i, axis=0)
            others = others[~np.isnan(others).any(axis=1)]
            if len(others) > 0:
                dists = np.sqrt(((others - centers[i]) ** 2).sum(axis=1))
                iso = float(dists.min()) / diag
            else:
                iso = np.nan
        else:
            iso = np.nan

        # --- elevation: top of mask's tight bbox, transformed as 1 - y/H ---
        x1, y1, x2, y2 = fig['bbox']
        elev = float(1 - (y1 / H)) if not np.isnan(y1) else np.nan

        # --- salience: sum over mask region (DIS), plus secondary high-coverage-50 ---
        mask = fig['mask']
        if np.any(mask):
            values = saliency_map[mask.astype(bool)]
            sal_sum = float(values.sum())
            sal_hc50 = float(np.mean(values >= 0.50))
        else:
            sal_sum = np.nan
            sal_hc50 = np.nan

        rows.append({
            'painting_id':              painting_id,
            'figure_id':                fig['figure_id'],
            'centroid_x':               fig['centroid_x'],
            'centroid_y':               fig['centroid_y'],
            'mask_area':                fig['mask_area'],
            'bbox_x1':                  x1, 'bbox_y1': y1, 'bbox_x2': x2, 'bbox_y2': y2,
            'iso_nearest_centroid_norm': iso,
            'elev_bbox_top':             elev,
            'sal_sum':                   sal_sum,
            'sal_high_coverage_50':      sal_hc50,
        })
    return pd.DataFrame(rows)

In [ ]:
# ---- batch loop over gate-passing paintings -------------------------------
all_feature_dfs = []
skipped = []

for painting_id in paintings_passed:
    image_path = IMAGES_DIR / f'{painting_id}.jpg'
    mask_dir   = SAM3_ROOT / painting_id
    dis_path   = find_dis_for_painting(DIS_DIR, painting_id)

    if not image_path.exists():
        skipped.append((painting_id, 'image missing'))
        continue
    if not mask_dir.exists() or not any(mask_dir.glob('*.png')):
        skipped.append((painting_id, 'no SAM3 masks'))
        continue
    if dis_path is None:
        skipped.append((painting_id, 'no DIS saliency map'))
        continue

    try:
        image_rgb = load_image_rgb(image_path)
        H, W = image_rgb.shape[:2]
        figures = extract_figures(mask_dir, (H, W))
        if not figures:
            skipped.append((painting_id, 'no usable masks'))
            continue
        saliency = load_saliency_map(dis_path, (H, W))
        df = compute_painting_features(painting_id, figures, saliency, (H, W))
        df.to_csv(FEATURE_DIR / f'{painting_id}_features.csv', index=False)
        all_feature_dfs.append(df)
    except Exception as exc:
        skipped.append((painting_id, f'error: {exc}'))
    finally:
        gc.collect()

features = pd.concat(all_feature_dfs, ignore_index=True) if all_feature_dfs else pd.DataFrame()
features['painting_id'] = features['painting_id'].astype(str)
features.to_csv(OUTPUTS / 'all_features_unannotated.csv', index=False)

print(f'Processed: {features["painting_id"].nunique() if len(features) else 0} paintings,',
      f'{len(features)} figures')
if skipped:
    print(f'\nSkipped ({len(skipped)}):')
    for pid, reason in skipped:
        print(f'  {pid}: {reason}')

# 3. Match VIA boxes to detected figures

Each hand-drawn box is meant to contain one important figure. The primary test is
whether a detected figure's centroid falls inside the hand-drawn box; if several
do, the one closest to the box centre wins; if none do, the nearest centroid is
used as a fallback. The first VIA box for each painting is treated as the main
figure; subsequent boxes are supporting important figures.

In [ ]:
via_important_boxes = gate_module.load_via_important_boxes(ANNOTATIONS)

def _point_in_box(px, py, box):
    x1, y1, x2, y2 = box
    return (x1 <= px <= x2) and (y1 <= py <= y2)

def _box_center(box):
    x1, y1, x2, y2 = box
    return (x1 + x2) / 2.0, (y1 + y2) / 2.0

important_by_painting = {}   # painting_id -> set of important figure_ids
match_log = []

for pid, group in features.groupby('painting_id'):
    vboxes = via_important_boxes.get(pid, [])
    figs = group.dropna(subset=['centroid_x', 'centroid_y'])
    fig_points = [
        (r['figure_id'], float(r['centroid_x']), float(r['centroid_y']))
        for _, r in figs.iterrows()
    ]
    important_ids = set()
    for i, vbox in enumerate(vboxes):
        if not fig_points:
            break
        bcx, bcy = _box_center(vbox)
        inside = [
            (fid, fx, fy) for (fid, fx, fy) in fig_points
            if _point_in_box(fx, fy, vbox)
        ]
        candidates = inside if inside else fig_points
        best_fid, fx, fy = min(
            candidates,
            key=lambda t: (t[1] - bcx) ** 2 + (t[2] - bcy) ** 2
        )
        important_ids.add(best_fid)
        match_log.append({
            'painting_id':    pid,
            'via_box_index':  i,
            'matched_figure': best_fid,
            'matched_by':     'centroid in box' if inside else 'nearest fallback',
        })
    important_by_painting[pid] = important_ids

features['is_important'] = features.apply(
    lambda r: r['figure_id'] in important_by_painting.get(str(r['painting_id']), set()),
    axis=1,
)

match_log_df = pd.DataFrame(match_log)
match_log_df.to_csv(OUTPUTS / 'via_to_figure_matches.csv', index=False)
features.to_csv(OUTPUTS / 'all_features_annotated.csv', index=False)

print(f'Marked is_important: {int(features["is_important"].sum())}')

fallback = match_log_df[match_log_df['matched_by'] == 'nearest fallback']
if len(fallback):
    print(f'\n{len(fallback)} box(es) matched by fallback (worth eyeballing):')
    display(fallback)

# 4. Build the composite score

Each individual feature is min-max normalised **within painting** (so figures are
compared only against others in the same image), then combined as

$$\text{combo\_saliency\_heavy} = 0.50 \cdot \text{sal} + 0.25 \cdot \text{iso} + 0.25 \cdot \text{elev}$$

All three features are oriented so that higher = more emphasis.

In [ ]:
def minmax_within_painting(df, col, higher_is_better=True):
    df = df.copy()
    out = f'norm_{col}'
    df[out] = np.nan
    for pid, g in df.groupby('painting_id'):
        v = g[col].astype(float)
        valid = v.notna()
        if not valid.any():
            continue
        vv = v[valid]
        if vv.max() == vv.min():
            normed = pd.Series(np.full(len(vv), 0.5), index=vv.index)
        else:
            normed = (vv - vv.min()) / (vv.max() - vv.min())
        if not higher_is_better:
            normed = 1 - normed
        df.loc[normed.index, out] = normed
    return df

features = minmax_within_painting(features, 'iso_nearest_centroid_norm', higher_is_better=True)
features = minmax_within_painting(features, 'elev_bbox_top',             higher_is_better=True)
features = minmax_within_painting(features, 'sal_sum',                   higher_is_better=True)

features['combo_saliency_heavy'] = (
    0.50 * features['norm_sal_sum']
  + 0.25 * features['norm_iso_nearest_centroid_norm']
  + 0.25 * features['norm_elev_bbox_top']
)

features.to_csv(OUTPUTS / 'all_features_with_composite.csv', index=False)
print('Composite combo_saliency_heavy built for', features["painting_id"].nunique(), 'paintings')

# 5. Per-painting summary

Sorted from highest YOLO-detected figure count to lowest. Reports, for every
painting that passed the gate:

- `n_yolo_figures` — figures the detector found (= figures we scored)
- `n_important_figs` — hand-drawn important figures (matched to a detected figure)
- `top_fig_id` — the figure ranked #1 by `combo_saliency_heavy`
- `top_is_important` — is that top-ranked figure an important figure? (per-painting Precision@1)
- `best_imp_combo` — the highest composite score among important figures
- `best_imp_sal_sum` — the highest raw `sal_sum` among important figures
- `n_important_in_top3` — how many important figures land in the top 3

In [ ]:
PRIMARY = 'combo_saliency_heavy'

rows = []
for pid, g in features.groupby('painting_id'):
    important = g[g['is_important']]
    if important.empty:
        continue   # no important figure matched - drop from summary

    # Rank by primary composite (descending)
    g_sorted = g.sort_values(PRIMARY, ascending=False).reset_index(drop=True)
    top_row = g_sorted.iloc[0]

    # Top-3 important hits
    top3_ids = set(g_sorted.head(3)['figure_id'])
    n_imp_top3 = int(sum(1 for fid in important['figure_id'] if fid in top3_ids))

    rows.append({
        'painting_id':         pid,
        'n_yolo_figures':      len(g),
        'n_important_figs':    len(important),
        'top_fig_id':          top_row['figure_id'],
        'top_is_important':    bool(top_row['is_important']),
        'best_imp_combo':      float(important[PRIMARY].max()),
        'best_imp_sal_sum':    float(important['sal_sum'].max()),
        'n_important_in_top3': n_imp_top3,
    })

summary = (pd.DataFrame(rows)
             .sort_values('n_yolo_figures', ascending=False)
             .reset_index(drop=True))

summary.to_csv(SUMMARY_DIR / 'per_painting_summary.csv', index=False)
print(f'Summary rows: {len(summary)}')
print(f'Saved: {SUMMARY_DIR / "per_painting_summary.csv"}')
display(summary.round(4))

# 6. Locked-in technique evaluation

Both locked-in techniques scored against the `is_important` ground-truth annotation,
using the same within-painting ranking metrics as the pilot:

- **Precision@1**: rank-1 figure is important
- **Recall@3**: fraction of a painting's important figures inside the top 3
- **MRR**: mean reciprocal rank of the best-ranked important figure (primary metric)
- **Hit@3**: any important figure inside the top 3

In [ ]:
def _ranks_with_ties(scores):
    scores = np.asarray(scores, dtype=float)
    order  = np.argsort(-scores, kind='mergesort')
    ranks  = np.empty(len(scores), dtype=float)
    i = 0
    while i < len(scores):
        j = i
        while j + 1 < len(scores) and scores[order[j + 1]] == scores[order[i]]:
            j += 1
        mean_rank = np.mean(np.arange(i + 1, j + 2))
        for k in range(i, j + 1):
            ranks[order[k]] = mean_rank
        i = j + 1
    return ranks


def rank_score(df, feature_col):
    p1, p3, r1, r3, mrr, h3 = [], [], [], [], [], []
    usable = 0
    for _, g in df.groupby('painting_id'):
        g = g.dropna(subset=[feature_col, 'is_important'])
        if g.empty:
            continue
        imp = g['is_important'].to_numpy(dtype=bool)
        if imp.sum() == 0 or (~imp).sum() == 0:
            continue
        vals  = g[feature_col].to_numpy(dtype=float)
        ranks = _ranks_with_ties(vals)
        n_imp = int(imp.sum())
        n     = len(ranks)

        p1.append(float(imp[ranks <= 1].sum()) / min(1, n))
        p3.append(float(imp[ranks <= 3].sum()) / min(3, n))
        r1.append(int(np.sum(ranks[imp] <= 1)) / n_imp)
        r3.append(int(np.sum(ranks[imp] <= 3)) / n_imp)
        mrr.append(1.0 / ranks[imp].min())
        h3.append(float(np.any(ranks[imp] <= 3)))
        usable += 1

    if usable == 0:
        return {k: np.nan for k in
                ['usable_paintings', 'P@1', 'P@3', 'R@1', 'R@3', 'MRR', 'Hit@3']}
    return {
        "usable_paintings": usable,
        "P@1":   float(np.mean(p1)),
        "P@3":   float(np.mean(p3)),
        "R@1":   float(np.mean(r1)),
        "R@3":   float(np.mean(r3)),
        "MRR":   float(np.mean(mrr)),
        "Hit@3": float(np.mean(h3)),
    }


def random_baseline(df, n_sims=1000, seed=42):
    rng = np.random.default_rng(seed=seed)
    p1_r, p3_r, r1_r, r3_r, mrr_r, h3_r = [], [], [], [], [], []
    usable = 0
    for _, g in df.groupby('painting_id'):
        imp   = g['is_important'].to_numpy(dtype=bool)
        n     = len(imp)
        n_imp = int(imp.sum())
        if n_imp == 0 or n_imp == n:
            continue
        sims = []
        for _ in range(n_sims):
            perm  = rng.permutation(n)
            ranks = np.argsort(perm) + 1
            sims.append((
                imp[ranks <= 1].sum() / 1,
                imp[ranks <= 3].sum() / min(3, n),
                int(np.sum(ranks[imp] <= 1)) / n_imp,
                int(np.sum(ranks[imp] <= 3)) / n_imp,
                1.0 / ranks[imp].min(),
                float(np.any(ranks[imp] <= 3)),
            ))
        sims = np.array(sims, dtype=float)
        p1_r.append(sims[:, 0].mean())
        p3_r.append(sims[:, 1].mean())
        r1_r.append(sims[:, 2].mean())
        r3_r.append(sims[:, 3].mean())
        mrr_r.append(sims[:, 4].mean())
        h3_r.append(sims[:, 5].mean())
        usable += 1

    return {
        "usable_paintings": usable,
        "P@1":   float(np.mean(p1_r)),
        "P@3":   float(np.mean(p3_r)),
        "R@1":   float(np.mean(r1_r)),
        "R@3":   float(np.mean(r3_r)),
        "MRR":   float(np.mean(mrr_r)),
        "Hit@3": float(np.mean(h3_r)),
    }


results = pd.DataFrame([
    {'Technique': 'Full composite',    **rank_score(features, 'combo_saliency_heavy')},
    {'Technique': 'Salience baseline', **rank_score(features, 'sal_high_coverage_50')},
    {'Technique': 'Random baseline',   **random_baseline(features)},
])

cols = ['Technique', 'usable_paintings', 'P@1', 'P@3', 'R@1', 'R@3', 'MRR', 'Hit@3']
results = results[cols]

results.to_csv(SUMMARY_DIR / 'comparison_table.csv', index=False)
print('=' * 78)
print('COMPARISON TABLE — pipeline vs. single feature vs. random (is_important)')
print('=' * 78)
display(results.round(3))

# 7. Extra Analysis and visualisation

In [ ]:
# ── ABLATION TABLE ────────────────────────────────────────────────────────────
# Tests marginal contribution of each cue.
# Uses the same rank_score() function defined in Cell 19.
# Requires the composite columns built in Cell 15.

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Build pair composites if not already present
# (full composite is already built in Cell 15 as combo_saliency_heavy)
for col in ['norm_iso_nearest_centroid_norm', 'norm_elev_bbox_top', 'norm_sal_sum']:
    if col not in features.columns:
        raise ValueError(f"Column {col} missing — run Cell 15 first.")

# Salience + Isolation pair
if 'combo_iso_sal' not in features.columns:
    features['combo_iso_sal'] = (
        0.50 * features['norm_sal_sum']
      + 0.50 * features['norm_iso_nearest_centroid_norm']
    )

# Salience + Elevation pair
if 'combo_elev_sal' not in features.columns:
    features['combo_elev_sal'] = (
        0.50 * features['norm_sal_sum']
      + 0.50 * features['norm_elev_bbox_top']
    )

# Isolation + Elevation pair
if 'combo_iso_elev' not in features.columns:
    features['combo_iso_elev'] = (
        0.50 * features['norm_iso_nearest_centroid_norm']
      + 0.50 * features['norm_elev_bbox_top']
    )

ablation_specs = [
    ('sal_sum',               'Salience only'),
    ('iso_nearest_centroid_norm', 'Isolation only'),
    ('elev_bbox_top',         'Elevation only'),
    ('combo_iso_sal',         'Salience + Isolation'),
    ('combo_elev_sal',        'Salience + Elevation'),
    ('combo_iso_elev',        'Isolation + Elevation'),
    ('combo_saliency_heavy',  'Full composite (S+I+E)'),
]

abl_rows = []
for col, label in ablation_specs:
    if col not in features.columns:
        print(f"Skipping {col} — not in features")
        continue
    r = rank_score(features, col)
    r['Technique'] = label
    abl_rows.append(r)

abl_df = pd.DataFrame(abl_rows)[['Technique', 'usable_paintings',
                                   'P@1', 'P@3', 'R@1', 'R@3', 'MRR', 'Hit@3']]

print('=' * 78)
print('ABLATION — marginal contribution of each cue combination')
print('=' * 78)
display(abl_df.round(3))
abl_df.to_csv(SUMMARY_DIR / 'ablation_table.csv', index=False)
print(f'Saved: {SUMMARY_DIR / "ablation_table.csv"}')

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
x     = np.arange(len(abl_df))
width = 0.25
ax.bar(x - width, abl_df['P@1'].values,   width, label='P@1',   color='#2166ac')
ax.bar(x,         abl_df['MRR'].values,   width, label='MRR',   color='#4dac26')
ax.bar(x + width, abl_df['Hit@3'].values, width, label='Hit@3', color='#d01c8b')
ax.set_xticks(x)
ax.set_xticklabels(abl_df['Technique'].values, rotation=22, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.set_title('Ablation: P@1, MRR, Hit@3 by cue combination (full run)', fontsize=11)
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
abl_plot = SUMMARY_DIR / 'ablation_bar_chart.png'
plt.savefig(abl_plot, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {abl_plot}')

In [ ]:
# ── BOOTSTRAP CONFIDENCE INTERVALS ───────────────────────────────────────────
# Resamples paintings (with replacement) to compute 95% CIs for
# P@1, MRR, Hit@3 for the three main techniques.
# Non-overlapping CIs = composite reliably outperforms baseline.

def _per_painting_metric_vectors(df, feature_col):
    """
    Returns arrays of per-painting P@1, MRR, Hit@3 values.
    One entry per scorable painting (>=1 important, >=1 non-important).
    """
    p1_v, mrr_v, h3_v = [], [], []
    for _, g in df.groupby('painting_id'):
        g = g.dropna(subset=[feature_col, 'is_important'])
        if g.empty:
            continue
        imp = g['is_important'].to_numpy(dtype=bool)
        if imp.sum() == 0 or (~imp).sum() == 0:
            continue
        vals  = g[feature_col].to_numpy(dtype=float)
        ranks = _ranks_with_ties(vals)
        p1_v.append(float(imp[ranks <= 1].sum()))
        mrr_v.append(1.0 / ranks[imp].min())
        h3_v.append(float(np.any(ranks[imp] <= 3)))
    return np.array(p1_v), np.array(mrr_v), np.array(h3_v)


def _random_per_painting_vectors(df, n_sims=1000, seed=42):
    """Monte Carlo average per-painting P@1, MRR, Hit@3 for random baseline."""
    rng = np.random.default_rng(seed)
    p1_v, mrr_v, h3_v = [], [], []
    for _, g in df.groupby('painting_id'):
        imp = g['is_important'].to_numpy(dtype=bool)
        n, n_imp = len(imp), int(imp.sum())
        if n_imp == 0 or n_imp == n:
            continue
        sims = []
        for _ in range(n_sims):
            perm  = rng.permutation(n)
            ranks = np.argsort(perm) + 1
            sims.append([
                imp[ranks <= 1].sum(),
                1.0 / ranks[imp].min(),
                float(np.any(ranks[imp] <= 3)),
            ])
        sims = np.array(sims)
        p1_v.append(sims[:, 0].mean())
        mrr_v.append(sims[:, 1].mean())
        h3_v.append(sims[:, 2].mean())
    return np.array(p1_v), np.array(mrr_v), np.array(h3_v)


def bootstrap_ci(values, n_boot=5000, seed=42):
    rng = np.random.default_rng(seed)
    n   = len(values)
    boot = np.array([
        rng.choice(values, size=n, replace=True).mean()
        for _ in range(n_boot)
    ])
    return values.mean(), float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))


print('Computing bootstrap CIs (Monte Carlo for random baseline takes ~30s)...')

specs = [
    ('Full composite',    *_per_painting_metric_vectors(features, 'combo_saliency_heavy')),
    ('Salience baseline', *_per_painting_metric_vectors(features, 'sal_high_coverage_50')),
    ('Random baseline',   *_random_per_painting_vectors(features)),
]

ci_rows = []
for label, p1_v, mrr_v, h3_v in specs:
    row = {'Technique': label, 'n': len(p1_v)}
    for metric, vals in [('P@1', p1_v), ('MRR', mrr_v), ('Hit@3', h3_v)]:
        mean, lo, hi = bootstrap_ci(vals)
        row[metric]        = round(mean, 3)
        row[f'{metric} CI'] = f'[{lo:.3f}, {hi:.3f}]'
    ci_rows.append(row)

ci_df = pd.DataFrame(ci_rows)
print('\n' + '=' * 70)
print('BOOTSTRAP 95% CONFIDENCE INTERVALS — full run (n=204 scorable paintings)')
print('=' * 70)
display(ci_df)
ci_df.to_csv(SUMMARY_DIR / 'bootstrap_ci.csv', index=False)
print(f'Saved: {SUMMARY_DIR / "bootstrap_ci.csv"}')

# ── CI plot ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=False)
colors = ['#2166ac', '#4dac26', '#d01c8b']

for ax, metric in zip(axes, ['P@1', 'MRR', 'Hit@3']):
    means = ci_df[metric].values
    lows  = ci_df[f'{metric} CI'].apply(lambda s: float(s.strip('[]').split(',')[0])).values
    highs = ci_df[f'{metric} CI'].apply(lambda s: float(s.strip('[]').split(',')[1])).values
    labels = ci_df['Technique'].values
    x = np.arange(len(labels))

    ax.bar(x, means, color=colors, alpha=0.8, edgecolor='#333', zorder=2)
    ax.errorbar(x, means,
                yerr=[means - lows, highs - means],
                fmt='none', color='black', capsize=5, linewidth=1.5, zorder=3)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=8)
    ax.set_title(metric, fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('Bootstrap 95% CIs — full run', fontsize=11, y=1.02)
plt.tight_layout()
ci_plot = SUMMARY_DIR / 'bootstrap_ci_plot.png'
plt.savefig(ci_plot, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {ci_plot}')

In [ ]:
# ── VIOLIN PLOTS — feature distributions ─────────────────────────────────────
import matplotlib.patches as mpatches

PRIMARY_FEATURES = {
    'iso_nearest_centroid_norm': 'Isolation\n(nearest centroid, normalised)',
    'elev_bbox_top':             'Elevation\n(bbox top, 0=bottom 1=top)',
    'sal_sum':                   'Salience\n(total saliency in mask)',
}

# Scorable paintings only
scorable_pids = [
    pid for pid, g in features.groupby('painting_id')
    if g['is_important'].sum() > 0 and (~g['is_important']).sum() > 0
]
df_plot = features[features['painting_id'].isin(scorable_pids)].copy()

COLORS = {True: '#d01c8b', False: '#4dac26'}
fig, axes = plt.subplots(1, 3, figsize=(13, 5))

for ax, (feat, label) in zip(axes, PRIMARY_FEATURES.items()):
    if feat not in df_plot.columns:
        ax.set_title(f'{label}\n(not found)')
        continue
    data_imp    = df_plot.loc[df_plot['is_important'] == True,  feat].dropna().values
    data_notimp = df_plot.loc[df_plot['is_important'] == False, feat].dropna().values

    parts = ax.violinplot([data_notimp, data_imp], positions=[1, 2],
                          showmedians=True, showextrema=True)
    for body, color in zip(parts['bodies'], [COLORS[False], COLORS[True]]):
        body.set_facecolor(color)
        body.set_alpha(0.65)
    for key in ['cmedians', 'cbars', 'cmins', 'cmaxes']:
        if key in parts:
            parts[key].set_color('black')
            parts[key].set_linewidth(1.2)

    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Non-important', 'Important'], fontsize=9)
    ax.set_title(label, fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
    for pos, data in [(1, data_notimp), (2, data_imp)]:
        med = np.median(data)
        ax.text(pos, med, f'  {med:.2f}', va='center', fontsize=8)

imp_patch    = mpatches.Patch(color=COLORS[True],  alpha=0.65, label='Important')
notimp_patch = mpatches.Patch(color=COLORS[False], alpha=0.65, label='Non-important')
fig.legend(handles=[imp_patch, notimp_patch], loc='upper right', fontsize=9)
fig.suptitle(
    f'Feature distributions: important vs non-important figures\n'
    f'(n={len(df_plot)} figures, {len(scorable_pids)} scorable paintings)',
    fontsize=11, y=1.01
)
plt.tight_layout()
dist_path = SUMMARY_DIR / 'feature_distributions.png'
plt.savefig(dist_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {dist_path}')

In [ ]:
# ── CORRELATION MATRIX ────────────────────────────────────────────────────────
from scipy.stats import spearmanr

feat_labels = {
    'iso_nearest_centroid_norm': 'Isolation',
    'elev_bbox_top':             'Elevation',
    'sal_sum':                   'Salience',
}
feats = [f for f in feat_labels if f in features.columns]
df_corr = features[feats].dropna()
n_obs = len(df_corr)
labels = [feat_labels[f] for f in feats]

corr_matrix = np.eye(len(feats))
pval_matrix = np.zeros((len(feats), len(feats)))
for i in range(len(feats)):
    for j in range(len(feats)):
        if i != j:
            r, p = spearmanr(df_corr[feats[i]], df_corr[feats[j]])
            corr_matrix[i, j] = r
            pval_matrix[i, j] = p

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr_matrix, vmin=-1, vmax=1, cmap='RdBu_r')
plt.colorbar(im, ax=ax, label='Spearman r')
ax.set_xticks(range(len(feats)))
ax.set_yticks(range(len(feats)))
ax.set_xticklabels(labels, fontsize=10)
ax.set_yticklabels(labels, fontsize=10)
for i in range(len(feats)):
    for j in range(len(feats)):
        val  = corr_matrix[i, j]
        pval = pval_matrix[i, j]
        star = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
        txt  = f'{val:.2f}\n{star}' if i != j else '1.00'
        ax.text(j, i, txt, ha='center', va='center', fontsize=9,
                color='white' if abs(val) > 0.5 else 'black')
ax.set_title(f'Spearman correlations — primary cues\n(n={n_obs} figure detections)', fontsize=10)
plt.tight_layout()
corr_path = SUMMARY_DIR / 'cue_correlation_matrix.png'
plt.savefig(corr_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {corr_path}')
print(pd.DataFrame(corr_matrix.round(3), index=labels, columns=labels))

# 8. Outputs

Everything this notebook produces lives under `OUTPUTS`:

- `iou_per_painting.csv` — per-painting gate results
- `detection_failures.csv` — paintings that failed the gate
- `all_features_unannotated.csv` — per-figure features before annotation matching
- `via_to_figure_matches.csv` — how each VIA box was matched to a detected figure
- `all_features_annotated.csv` — features + `is_main`, `is_important`, `importance_level`
- `all_features_with_composite.csv` — final feature table with the composite score
- `features_per_painting/<id>_features.csv` — per-painting feature dumps
- `summary/per_painting_summary.csv` — Section 5 table
- `summary/locked_in_technique_results.csv` — Section 6 ranking metrics
- `summary/iou_distribution.png` — gate histogram